# Interpolation Case 01: Interpolation Benchmarking


This notebook benchmarks interpolation in a developmental atlas where the pretrained Navigo model was fit on observed stages spaced about one day apart, then evaluated on the intermediate stages that fall between those training observations. It combines intermediate-stage reconstruction, quantitative benchmark scoring, and shared-embedding visualization for Navigo and optional repo-local baselines.

Biological objective:
1. Reconstruct intermediate developmental stages from the adjacent observed stages used in the one-day-interval training setup.
2. Quantify whether each reconstructed stage recovers cell-type composition, DEG signatures, and overall expression geometry.
3. Place Navigo and optional baseline methods on a shared UMAP anchored by the neighboring observed stages and the real intermediate-stage ground truth.

This case uses release-local atlas and checkpoint assets shipped in this repository. Optional `moscot`, `random`, and `tigon` prediction files can be added under `docs/tutorials/resources/interpolation/benchmark_predictions/` for side-by-side visualization.


Import required packages and set deterministic plotting defaults.

In [ ]:

import math
import os
import random
import re
import sys
import warnings
from pathlib import Path

import anndata
if not hasattr(anndata, 'read') and hasattr(anndata, 'read_h5ad'):
    anndata.read = anndata.read_h5ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from IPython.display import display
from scipy import sparse, stats
from scipy.spatial.distance import cdist, jensenshannon, pdist
from sklearn.decomposition import PCA

plt.rcParams['font.family'] = 'Arial'
sc.settings.verbosity = 0
warnings.filterwarnings('ignore', category=FutureWarning)
%config InlineBackend.figure_format = 'retina'


Define shared helpers for path resolution, time-axis handling, interpolation, and benchmark scoring.

In [ ]:

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'docs' / 'tutorials').exists() and (p / 'navigo').exists():
            return p
    raise RuntimeError('Could not locate repository root containing docs/tutorials and the navigo package.')


def require_path(path_like, label: str) -> Path:
    path = Path(path_like)
    if not path.exists():
        raise FileNotFoundError(
            f'{label} not found: {path}. Update the configuration cell with a valid local path before running this notebook.'
        )
    return path


def read_h5ad(path_like):
    path = Path(path_like)
    if hasattr(anndata, 'read_h5ad'):
        return anndata.read_h5ad(path)
    return anndata.read(path)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


def to_dense(x) -> np.ndarray:
    return x.toarray() if sparse.issparse(x) else np.asarray(x)


def gene_names_from_var(var: pd.DataFrame) -> np.ndarray:
    if 'gene_name' in var.columns:
        return var['gene_name'].astype(str).to_numpy()
    return var.index.astype(str).to_numpy()


def parse_numeric_token(value) -> float:
    match = re.search(r'-?\d+(?:\.\d+)?', str(value))
    if match is None:
        raise ValueError(f'Could not parse a numeric stage value from {value!r}.')
    return float(match.group())


def prepare_time_axis(
    adata: anndata.AnnData,
    day_col: str = 'day',
    time_col: str = 'time',
    prefer_existing_time: bool = False,
):
    adata = adata.copy()

    if prefer_existing_time and time_col in adata.obs.columns:
        stage_values = pd.to_numeric(adata.obs[time_col], errors='raise').astype(float)
        if day_col in adata.obs.columns:
            stage_labels = adata.obs[day_col].astype(str)
        else:
            stage_labels = stage_values.map(lambda x: f'{x:g}')
            adata.obs[day_col] = stage_labels.values
        unique_stage = np.sort(stage_values.unique())
        model_times = np.asarray(unique_stage, dtype=float)
        stage_to_model = {float(stage): float(stage) for stage in unique_stage}
    elif day_col in adata.obs.columns:
        stage_values = adata.obs[day_col].astype(str).map(parse_numeric_token).astype(float)
        stage_labels = adata.obs[day_col].astype(str)
        unique_stage = np.sort(stage_values.unique())
        model_times = np.arange(len(unique_stage), dtype=float) * 0.5
        stage_to_model = {float(stage): float(model_time) for stage, model_time in zip(unique_stage, model_times)}
    elif time_col in adata.obs.columns:
        stage_values = pd.to_numeric(adata.obs[time_col], errors='raise').astype(float)
        stage_labels = stage_values.map(lambda x: f'{x:g}')
        adata.obs[day_col] = stage_labels.values
        unique_stage = np.sort(stage_values.unique())
        model_times = np.asarray(unique_stage, dtype=float)
        stage_to_model = {float(stage): float(stage) for stage in unique_stage}
    else:
        raise KeyError(f'AnnData must contain either obs[{day_col!r}] or obs[{time_col!r}].')

    stage_to_label = {}
    stage_array = stage_values.to_numpy(dtype=float)
    for stage in unique_stage:
        first_idx = int(np.flatnonzero(stage_array == stage)[0])
        stage_to_label[float(stage)] = str(stage_labels.iloc[first_idx])

    adata.obs['stage_value'] = stage_array
    adata.obs['time'] = stage_values.map(stage_to_model).to_numpy(dtype=float)
    adata.obs['day'] = stage_values.map(stage_to_label).astype(str).to_numpy()

    model_to_day = {stage_to_model[float(stage)]: stage_to_label[float(stage)] for stage in unique_stage}
    return adata, np.asarray(model_times, dtype=float), model_to_day


def get_norm_msmu(adata: anndata.AnnData):
    gene_names = gene_names_from_var(adata.var)

    if 'norm_msmu' in adata.obsm:
        msmu = np.asarray(adata.obsm['norm_msmu'], dtype=np.float32)
        return msmu, gene_names

    if 'Ms' in adata.layers and 'Mu' in adata.layers:
        ms = to_dense(adata.layers['Ms']).astype(np.float32)
        mu = to_dense(adata.layers['Mu']).astype(np.float32)
        msmu = np.concatenate([ms, mu], axis=1)
        denom = msmu.max(axis=0) - msmu.min(axis=0)
        msmu = (msmu - msmu.min(axis=0)) / (denom + 1e-7)
        return msmu.astype(np.float32), gene_names

    x = to_dense(adata.X).astype(np.float32)
    if x.shape[1] == adata.n_vars * 2:
        return x, gene_names

    raise ValueError('Could not recover a normalized Ms/Mu matrix from the AnnData object.')


def collapse_msmu(msmu: np.ndarray) -> np.ndarray:
    split = msmu.shape[1] // 2
    return msmu[:, :split] + msmu[:, split:]


def extract_gene_expression(adata: anndata.AnnData, reference_genes=None):
    gene_names = gene_names_from_var(adata.var)

    try:
        msmu, gene_names = get_norm_msmu(adata)
        expr = collapse_msmu(msmu)
    except ValueError:
        expr = to_dense(adata.X).astype(np.float32)
        if expr.shape[1] == adata.n_vars * 2:
            expr = collapse_msmu(expr)
        elif reference_genes is not None and expr.shape[1] == len(reference_genes) * 2:
            expr = collapse_msmu(expr)
        elif expr.shape[1] != adata.n_vars and reference_genes is None:
            raise

    if reference_genes is None:
        return expr, np.asarray(gene_names)

    reference_genes = np.asarray(reference_genes)
    if expr.shape[1] == len(reference_genes) and np.array_equal(gene_names[: len(reference_genes)], reference_genes):
        return expr, reference_genes
    if expr.shape[1] == len(reference_genes) and len(gene_names) != len(reference_genes):
        return expr, reference_genes

    gene_to_idx = {gene: idx for idx, gene in enumerate(np.asarray(gene_names))}
    overlap = sum(gene in gene_to_idx for gene in reference_genes)
    if overlap == 0:
        if expr.shape[1] == len(reference_genes):
            return expr, reference_genes
        raise ValueError('Could not align expression matrix to the reference gene order.')

    aligned = np.zeros((expr.shape[0], len(reference_genes)), dtype=expr.dtype)
    for j, gene in enumerate(reference_genes):
        idx = gene_to_idx.get(gene)
        if idx is not None:
            aligned[:, j] = expr[:, idx]
    return aligned, reference_genes


def interior_test_times(all_times: np.ndarray, train_times: np.ndarray) -> list[float]:
    train_set = {float(x) for x in train_times}
    valid = []
    for t in map(float, all_times):
        if t in train_set:
            continue
        if np.any(train_times < t) and np.any(train_times > t):
            valid.append(float(t))
    return valid


def neighboring_train_times(test_time: float, train_times: np.ndarray):
    prev_train = float(train_times[train_times < test_time].max())
    next_train = float(train_times[train_times > test_time].min())
    return prev_train, next_train


def transfer_labels(pred_matrix: np.ndarray, ref_matrix: np.ndarray, ref_obs: pd.DataFrame, columns, chunk_size: int = 2048):
    if not columns:
        return pd.DataFrame(index=np.arange(pred_matrix.shape[0]))

    nearest = np.zeros(pred_matrix.shape[0], dtype=int)
    for start in range(0, pred_matrix.shape[0], chunk_size):
        stop = min(pred_matrix.shape[0], start + chunk_size)
        dist = cdist(pred_matrix[start:stop], ref_matrix)
        nearest[start:stop] = np.argmin(dist, axis=1)
    return ref_obs.iloc[nearest][list(columns)].reset_index(drop=True)


def _bh_fdr(pvals: np.ndarray) -> np.ndarray:
    pvals = np.asarray(pvals, dtype=float)
    n = pvals.size
    order = np.argsort(pvals)
    ranked = pvals[order]
    adj_ranked = np.empty(n, dtype=float)
    prev = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = min(prev, ranked[i] * n / rank)
        adj_ranked[i] = val
        prev = val
    adjusted = np.empty(n, dtype=float)
    adjusted[order] = np.clip(adj_ranked, 0.0, 1.0)
    return adjusted


def wilcoxon_deg(x_target: np.ndarray, x_other: np.ndarray, gene_names: np.ndarray) -> pd.DataFrame:
    mean_target = x_target.mean(axis=0)
    mean_other = x_other.mean(axis=0)
    logfc = np.log2((mean_target + 1e-9) / (mean_other + 1e-9))

    pvals = np.ones(x_target.shape[1], dtype=float)
    scores = np.zeros(x_target.shape[1], dtype=float)
    for j in range(x_target.shape[1]):
        a = x_target[:, j]
        b = x_other[:, j]
        if np.allclose(a, a[0]) and np.allclose(b, b[0]) and np.isclose(a[0], b[0]):
            continue
        res = stats.mannwhitneyu(a, b, alternative='two-sided', method='asymptotic')
        pvals[j] = res.pvalue
        scores[j] = res.statistic

    df = pd.DataFrame(
        {
            'names': gene_names,
            'scores': scores,
            'logfoldchanges': logfc,
            'pvals': pvals,
            'pvals_adj': _bh_fdr(pvals),
        }
    )
    return df.sort_values(['pvals_adj', 'pvals', 'logfoldchanges'], ascending=[True, True, False]).reset_index(drop=True)


def signature_overlap(real_deg: pd.DataFrame, pred_deg: pd.DataFrame, direction: str, top_n: int = 100, p_adj: float = 0.05) -> float:
    if direction == 'up':
        real = real_deg[(real_deg['pvals_adj'] < p_adj) & (real_deg['logfoldchanges'] > 0)].nlargest(top_n, 'logfoldchanges')
        pred = pred_deg[(pred_deg['pvals_adj'] < p_adj) & (pred_deg['logfoldchanges'] > 0)].nlargest(top_n, 'logfoldchanges')
    elif direction == 'down':
        real = real_deg[(real_deg['pvals_adj'] < p_adj) & (real_deg['logfoldchanges'] < 0)].nsmallest(top_n, 'logfoldchanges')
        pred = pred_deg[(pred_deg['pvals_adj'] < p_adj) & (pred_deg['logfoldchanges'] < 0)].nsmallest(top_n, 'logfoldchanges')
    else:
        raise ValueError(f'Unsupported direction: {direction}')

    real_set = set(real['names'].astype(str))
    pred_set = set(pred['names'].astype(str))
    if not real_set or not pred_set:
        return float('nan')
    return len(real_set & pred_set) / min(len(real_set), len(pred_set))


def distribution_metrics(real_x: np.ndarray, pred_x: np.ndarray, seed: int = 0, max_cells: int = 1000, n_components: int = 20):
    rng = np.random.default_rng(seed)

    if real_x.shape[0] > max_cells:
        real_x = real_x[rng.choice(real_x.shape[0], size=max_cells, replace=False)]
    if pred_x.shape[0] > max_cells:
        pred_x = pred_x[rng.choice(pred_x.shape[0], size=max_cells, replace=False)]

    merged = np.vstack([real_x, pred_x])
    n_components = min(n_components, merged.shape[0] - 1, merged.shape[1])
    if n_components >= 2:
        pca = PCA(n_components=n_components, random_state=seed)
        merged = pca.fit_transform(merged)
        real_proj = merged[: real_x.shape[0]]
        pred_proj = merged[real_x.shape[0] :]
    else:
        real_proj = real_x
        pred_proj = pred_x

    if real_proj.shape[0] < 2 or pred_proj.shape[0] < 2:
        return {
            'wasserstein_distance': float('nan'),
            'mmd': float('nan'),
            'energy_distance': float('nan'),
        }

    n_proj = min(32, max(real_proj.shape[1], 1))
    projections = rng.normal(size=(n_proj, real_proj.shape[1]))
    projections /= np.linalg.norm(projections, axis=1, keepdims=True) + 1e-12
    w_dists = [stats.wasserstein_distance(real_proj @ vec, pred_proj @ vec) for vec in projections]
    wasserstein = float(np.mean(w_dists))

    pairwise = pdist(np.vstack([real_proj, pred_proj]))
    scale = float(np.median(pairwise[pairwise > 0])) if np.any(pairwise > 0) else 1.0
    gamma = 1.0 / (2.0 * scale * scale + 1e-12)

    xx = cdist(real_proj, real_proj, metric='sqeuclidean')
    yy = cdist(pred_proj, pred_proj, metric='sqeuclidean')
    xy = cdist(real_proj, pred_proj, metric='sqeuclidean')
    mmd = float(np.exp(-gamma * xx).mean() + np.exp(-gamma * yy).mean() - 2.0 * np.exp(-gamma * xy).mean())
    energy = float(2.0 * cdist(real_proj, pred_proj).mean() - cdist(real_proj, real_proj).mean() - cdist(pred_proj, pred_proj).mean())
    return {
        'wasserstein_distance': wasserstein,
        'mmd': mmd,
        'energy_distance': energy,
    }


def cell_type_metrics(real_obs: pd.DataFrame, pred_obs: pd.DataFrame, real_col: str = 'cell_type', pred_col: str = 'predicted_cell_type'):
    if real_col not in real_obs.columns or pred_col not in pred_obs.columns:
        return {'js_divergence': float('nan'), 'l1_distance': float('nan')}

    real_counts = real_obs[real_col].astype(str).value_counts(normalize=True)
    pred_counts = pred_obs[pred_col].astype(str).value_counts(normalize=True)
    categories = sorted(set(real_counts.index) | set(pred_counts.index))
    p = np.array([real_counts.get(cat, 0.0) for cat in categories], dtype=float)
    q = np.array([pred_counts.get(cat, 0.0) for cat in categories], dtype=float)
    return {
        'js_divergence': float(jensenshannon(p, q, base=2.0) ** 2),
        'l1_distance': float(np.abs(p - q).sum()),
    }


def subset_for_time(adata: anndata.AnnData, time_value: float, max_cells: int, seed: int, label: str):
    mask = np.isclose(adata.obs['time'].to_numpy(dtype=float), float(time_value))
    subset = adata[mask].copy()
    if subset.n_obs == 0:
        raise ValueError(f'No cells found for time={time_value} in {label}.')
    if subset.n_obs > max_cells:
        rng = np.random.default_rng(seed)
        keep = rng.choice(subset.n_obs, size=max_cells, replace=False)
        subset = subset[keep].copy()
    return subset


Set paths and runtime configuration for this held-out interpolation benchmark.

In [ ]:
repo_root = find_repo_root(Path.cwd())
tutorials_root = repo_root / 'docs' / 'tutorials'
resources_dir = tutorials_root / 'resources' / 'interpolation'
output_root = resources_dir / 'outputs' / '04_subtimediff1_benchmark'
table_dir = output_root / '01_tables'
figure_dir = output_root / '02_figures'

for directory in [output_root, table_dir, figure_dir]:
    directory.mkdir(parents=True, exist_ok=True)

INPUT_DATA_REL = Path('data/interpolation/subtimediff1_hgv.h5ad')
CHECKPOINT_REL = Path('checkpoints/interpolation/subtimediff1_checkpoint_round10.pth')
BENCHMARK_PREDICTION_DIR_REL = Path('docs/tutorials/resources/interpolation/benchmark_predictions')
benchmark_prediction_dir = repo_root / BENCHMARK_PREDICTION_DIR_REL
benchmark_prediction_dir.mkdir(parents=True, exist_ok=True)

INPUT_DATA = repo_root / INPUT_DATA_REL
CHECKPOINT_PATH = repo_root / CHECKPOINT_REL

HIDDEN_1 = 5012
HIDDEN_2 = 5012
FLOW_STEPS = 100
SEED = 42
ALPHA = 0.5
INFER_MODE = 'pred_average'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SCORE_TABLE_PATH = table_dir / '01_inference_score_table.csv'
METRICS_TABLE_PATH = table_dir / '02_benchmark_metrics.csv'
UMAP_FIG_PATH = figure_dir / '01_shared_umap_method_comparison.png'
EXISTING_NAVIGO_PREDICTION = benchmark_prediction_dir / f'subtimediff1_adata_navigo_scale_{INFER_MODE}.h5ad'

OPTIONAL_BENCHMARK_PREDICTIONS = {
    'moscot': benchmark_prediction_dir / 'subtimediff1_adata_moscot_scale.h5ad',
    'random': benchmark_prediction_dir / 'subtimediff1_adata_random_scale.h5ad',
    'tigon': benchmark_prediction_dir / 'subtimediff1_adata_tigon_scale.h5ad',
}

FOCUS_TIME = 5.5
MAX_UMAP_CELLS_PER_GROUP = 3000
MAX_SCORE_CELLS = 512
MAX_DEG_CELLS = 1024

print('Repository root: .')
print(f'Input data: {INPUT_DATA.relative_to(repo_root)}')
print(f'Checkpoint: {CHECKPOINT_PATH.relative_to(repo_root)}')
print(f'Notebook outputs: {output_root.relative_to(repo_root)}')


## Step 1: Reconstruct intermediate stages with a pretrained Navigo model
We use a pretrained model fit on developmental stages sampled at roughly one-day intervals, then interpolate the intermediate stages between each pair of neighboring training stages. The reconstructed `pred_adata` object stays in memory for the downstream metric and UMAP cells, while summary tables are still written under the interpolation outputs directory.


In [ ]:

require_path(INPUT_DATA, 'INPUT_DATA')
set_seed(SEED)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from navigo.distance import earth_mover_distance
from navigo.model import MLPTimeGRN, Navigo
from navigo.utils import calculate_distance

atlas_backed = anndata.read_h5ad(INPUT_DATA, backed='r')
atlas_var = atlas_backed.var.copy()
gene_names = gene_names_from_var(atlas_var)
atlas_var.index = pd.Index(gene_names, dtype=str)
atlas_var.index.name = None

atlas_meta = anndata.AnnData(obs=atlas_backed.obs.copy(), var=atlas_var.copy())
atlas_meta, all_times, model_to_day = prepare_time_axis(atlas_meta)
atlas_obs = atlas_meta.obs.copy()
time_values = atlas_obs['time'].to_numpy(dtype=float)
train_times = all_times[::2]
test_times = interior_test_times(all_times, train_times)
atlas_msmu_dim = atlas_backed.obsm['norm_msmu'].shape[1]

def sample_index_array(index_array, max_cells, seed):
    index_array = np.asarray(index_array, dtype=int)
    if len(index_array) <= max_cells:
        return index_array
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(index_array, size=max_cells, replace=False))

def load_atlas_msmu_rows(row_idx):
    row_idx = np.asarray(row_idx, dtype=int)
    if row_idx.size == 0:
        return np.zeros((0, atlas_msmu_dim), dtype=np.float32)
    order = np.argsort(row_idx, kind='stable')
    sorted_idx = row_idx[order]
    unique_idx, inverse = np.unique(sorted_idx, return_inverse=True)
    sorted_rows = np.asarray(atlas_backed.obsm['norm_msmu'][unique_idx], dtype=np.float32)[inverse]
    restored = np.empty_like(sorted_rows)
    restored[order] = sorted_rows
    return restored

def load_atlas_expr_rows(row_idx):
    return collapse_msmu(load_atlas_msmu_rows(row_idx))

def load_atlas_subset_for_time(time_value, max_cells, seed, label):
    mask_idx = np.flatnonzero(np.isclose(time_values, float(time_value)))
    if mask_idx.size == 0:
        raise ValueError(f'No cells found for time={time_value} in {label}.')
    picked_idx = sample_index_array(mask_idx, max_cells=max_cells, seed=seed)
    subset_msmu = load_atlas_msmu_rows(picked_idx)
    subset_expr = collapse_msmu(subset_msmu)
    subset_obs = atlas_obs.iloc[picked_idx].copy().reset_index(drop=True)
    subset = anndata.AnnData(X=subset_expr, obs=subset_obs, var=atlas_var.copy())
    subset.obsm['norm_msmu'] = subset_msmu
    return subset

pred_blocks = []
pred_expr_blocks = []
pred_obs_frames = []
score_rows = []

if EXISTING_NAVIGO_PREDICTION.exists() and INFER_MODE == 'gt_average':
    pred_raw = read_h5ad(EXISTING_NAVIGO_PREDICTION)
    pred_raw, _, _ = prepare_time_axis(pred_raw, prefer_existing_time=True)
    pred_time_values = pred_raw.obs['time'].to_numpy(dtype=float)
    pred_msmu = np.asarray(pred_raw.X, dtype=np.float32)
    pred_expr_all = collapse_msmu(pred_msmu)

    if pred_msmu.shape[1] != atlas_msmu_dim:
        raise ValueError(
            f'Existing prediction width {pred_msmu.shape[1]} does not match atlas width {atlas_msmu_dim}.'
        )

    pred_obs = pred_raw.obs.copy().reset_index(drop=True)
    pred_obs['day'] = pred_obs['time'].map(model_to_day)
    pred_obs['source_time'] = np.nan
    pred_obs['source_day'] = None
    pred_obs['dest_time'] = np.nan
    pred_obs['dest_day'] = None

    for offset, test_time in enumerate(test_times):
        prev_train, next_train = neighboring_train_times(test_time, train_times)
        target_idx = np.flatnonzero(np.isclose(time_values, test_time))
        source_idx = np.flatnonzero(np.isclose(time_values, prev_train))
        dest_idx = np.flatnonzero(np.isclose(time_values, next_train))
        pred_idx = np.flatnonzero(np.isclose(pred_time_values, test_time))
        if len(target_idx) == 0 or len(source_idx) == 0 or len(dest_idx) == 0 or len(pred_idx) == 0:
            continue

        pred_obs.loc[pred_idx, 'source_time'] = float(prev_train)
        pred_obs.loc[pred_idx, 'source_day'] = model_to_day[float(prev_train)]
        pred_obs.loc[pred_idx, 'dest_time'] = float(next_train)
        pred_obs.loc[pred_idx, 'dest_day'] = model_to_day[float(next_train)]

        score_target_idx = sample_index_array(target_idx, max_cells=MAX_SCORE_CELLS, seed=SEED + 1000 + offset)
        score_source_idx = sample_index_array(source_idx, max_cells=MAX_SCORE_CELLS, seed=SEED + 2000 + offset)
        score_dest_idx = sample_index_array(dest_idx, max_cells=MAX_SCORE_CELLS, seed=SEED + 3000 + offset)
        score_pred_idx = sample_index_array(pred_idx, max_cells=MAX_SCORE_CELLS, seed=SEED + 4000 + offset)

        target_expr = load_atlas_expr_rows(score_target_idx)
        baseline_expr = load_atlas_expr_rows(score_source_idx)
        future_expr = load_atlas_expr_rows(score_dest_idx)
        pred_expr = pred_expr_all[score_pred_idx]

        score_rows.append(
            {
                'time': float(test_time),
                'day': model_to_day[float(test_time)],
                'source_time': float(prev_train),
                'dest_time': float(next_train),
                'n_pred_cells': int(len(pred_idx)),
                'score_sample_cells': int(len(score_pred_idx)),
                'emd_prediction': float(earth_mover_distance(target_expr, pred_expr)),
                'emd_baseline_source': float(earth_mover_distance(target_expr, baseline_expr)),
                'emd_future_stage': float(earth_mover_distance(target_expr, future_expr)),
            }
        )

    pred_obs.index = [f'pred_{i}' for i in range(len(pred_obs))]
    pred_adata = anndata.AnnData(X=pred_expr_all, obs=pred_obs, var=atlas_var.copy())
    pred_adata.obsm['norm_msmu'] = pred_msmu
    pred_adata.uns['inference_mode'] = INFER_MODE
    pred_adata.uns['alpha'] = ALPHA
    pred_adata.uns['flow_steps'] = FLOW_STEPS
    pred_adata.uns['source_prediction_path'] = str(EXISTING_NAVIGO_PREDICTION)
else:
    require_path(CHECKPOINT_PATH, 'CHECKPOINT_PATH')
    model = MLPTimeGRN(input_dim=atlas_msmu_dim, hidden_1=HIDDEN_1, hidden_2=HIDDEN_2).to(DEVICE)
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
    model.eval()
    navigo = Navigo(model=model, num_steps=FLOW_STEPS, device=DEVICE)

    for offset, test_time in enumerate(test_times):
        prev_train, next_train = neighboring_train_times(test_time, train_times)
        source_idx = np.flatnonzero(np.isclose(time_values, prev_train))
        dest_idx = np.flatnonzero(np.isclose(time_values, next_train))
        target_idx = np.flatnonzero(np.isclose(time_values, test_time))
        if len(source_idx) == 0 or len(dest_idx) == 0 or len(target_idx) == 0:
            continue

        rng = np.random.default_rng(SEED + offset)
        sample_size = len(target_idx)
        source_sample_idx = rng.choice(source_idx, size=sample_size, replace=True)
        dest_sample_idx = rng.choice(dest_idx, size=sample_size, replace=True)

        z0 = torch.tensor(load_atlas_msmu_rows(source_sample_idx), dtype=torch.float32)
        z1 = torch.tensor(load_atlas_msmu_rows(dest_sample_idx), dtype=torch.float32)
        t0 = torch.full((sample_size,), float(prev_train), dtype=torch.float32, device=DEVICE)
        t1 = torch.full((sample_size,), float(next_train), dtype=torch.float32, device=DEVICE)
        ti = torch.full((sample_size,), float(test_time), dtype=torch.float32, device=DEVICE)

        pred_forward = navigo.sample_ode_time_interval(z_full=z0, t_start=t0, t_end=t1, N=FLOW_STEPS)
        pred_forward_half = navigo.sample_ode_time_interval(z_full=z0, t_start=t0, t_end=ti, N=FLOW_STEPS)
        pred_backward_half = navigo.sample_ode_time_interval(z_full=z1, t_start=t1, t_end=ti, N=FLOW_STEPS)

        distances = calculate_distance(
            torch.tensor(pred_forward, dtype=torch.float32, device=DEVICE),
            z1.to(DEVICE),
        )
        alignment = torch.argmin(distances, dim=1).detach().cpu().numpy()

        if INFER_MODE == 'forward':
            pred_mid = pred_forward_half
        elif INFER_MODE == 'backward':
            pred_mid = pred_backward_half
        elif INFER_MODE == 'pred_average':
            pred_mid = (1.0 - ALPHA) * pred_backward_half[alignment] + ALPHA * pred_forward_half
        elif INFER_MODE == 'gt_average':
            pred_mid = (1.0 - ALPHA) * z1.detach().cpu().numpy()[alignment] + ALPHA * z0.detach().cpu().numpy()
        else:
            raise ValueError(f'Unsupported INFER_MODE: {INFER_MODE}')

        pred_expr = collapse_msmu(pred_mid)
        score_target_idx = sample_index_array(target_idx, max_cells=MAX_SCORE_CELLS, seed=SEED + 1000 + offset)
        score_source_idx = sample_index_array(source_idx, max_cells=MAX_SCORE_CELLS, seed=SEED + 2000 + offset)
        score_dest_idx = sample_index_array(dest_idx, max_cells=MAX_SCORE_CELLS, seed=SEED + 3000 + offset)
        score_pred_idx = sample_index_array(np.arange(pred_expr.shape[0]), max_cells=MAX_SCORE_CELLS, seed=SEED + 4000 + offset)

        score_rows.append(
            {
                'time': float(test_time),
                'day': model_to_day[float(test_time)],
                'source_time': float(prev_train),
                'dest_time': float(next_train),
                'n_pred_cells': int(sample_size),
                'score_sample_cells': int(len(score_pred_idx)),
                'emd_prediction': float(earth_mover_distance(load_atlas_expr_rows(score_target_idx), pred_expr[score_pred_idx])),
                'emd_baseline_source': float(earth_mover_distance(load_atlas_expr_rows(score_target_idx), load_atlas_expr_rows(score_source_idx))),
                'emd_future_stage': float(earth_mover_distance(load_atlas_expr_rows(score_target_idx), load_atlas_expr_rows(score_dest_idx))),
            }
        )

        obs_frame = pd.DataFrame(
            {
                'time': np.repeat(float(test_time), sample_size),
                'day': np.repeat(model_to_day[float(test_time)], sample_size),
                'source_time': np.repeat(float(prev_train), sample_size),
                'source_day': np.repeat(model_to_day[float(prev_train)], sample_size),
                'dest_time': np.repeat(float(next_train), sample_size),
                'dest_day': np.repeat(model_to_day[float(next_train)], sample_size),
            }
        )
        pred_blocks.append(pred_mid.astype(np.float32))
        pred_expr_blocks.append(pred_expr.astype(np.float32))
        pred_obs_frames.append(obs_frame)

    if not pred_blocks:
        raise RuntimeError('No held-out interpolation predictions were generated. Check the time axis in the input atlas.')

    pred_obs = pd.concat(pred_obs_frames, ignore_index=True)
    pred_obs.index = [f'pred_{i}' for i in range(len(pred_obs))]
    pred_adata = anndata.AnnData(X=np.vstack(pred_expr_blocks), obs=pred_obs, var=atlas_var.copy())
    pred_adata.obsm['norm_msmu'] = np.vstack(pred_blocks)
    pred_adata.uns['inference_mode'] = INFER_MODE
    pred_adata.uns['alpha'] = ALPHA
    pred_adata.uns['flow_steps'] = FLOW_STEPS

if pred_adata.n_obs == 0:
    raise RuntimeError('No held-out interpolation predictions are available for downstream evaluation.')

score_df = pd.DataFrame(score_rows).sort_values('time').reset_index(drop=True)
score_df.to_csv(SCORE_TABLE_PATH, index=False)

display(score_df)
print(f"Prepared prediction object with {pred_adata.n_obs} cells across {pred_adata.obs['day'].nunique()} intermediate evaluation stages.")



## Step 2: Quantify recovery of cell-type, DEG, and distribution structure
For each intermediate evaluation stage, we compare the predicted population against the real cells from that developmental time point. Cell-type agreement is measured on transferred annotations, DEG overlap is computed against the neighboring-stage background, and global geometry is summarized with sliced Wasserstein distance, RBF MMD, and energy distance.


In [ ]:
pred_time_values = pred_adata.obs['time'].to_numpy(dtype=float)

metrics_rows = []
for offset, test_time in enumerate(test_times):
    prev_train, next_train = neighboring_train_times(test_time, train_times)
    real_window_idx = np.flatnonzero(
        np.isclose(time_values, prev_train) | np.isclose(time_values, test_time) | np.isclose(time_values, next_train)
    )
    real_target_idx = np.flatnonzero(np.isclose(time_values, test_time))
    real_other_idx = real_window_idx[~np.isin(real_window_idx, real_target_idx)]
    pred_target_idx = np.flatnonzero(np.isclose(pred_time_values, test_time))

    if real_target_idx.size == 0 or real_other_idx.size == 0 or pred_target_idx.size == 0:
        continue

    sampled_target_idx = sample_index_array(real_target_idx, max_cells=MAX_DEG_CELLS, seed=SEED + 5000 + offset)
    sampled_other_idx = sample_index_array(real_other_idx, max_cells=MAX_DEG_CELLS, seed=SEED + 6000 + offset)
    sampled_pred_idx = sample_index_array(pred_target_idx, max_cells=MAX_DEG_CELLS, seed=SEED + 7000 + offset)

    real_target_expr = load_atlas_expr_rows(sampled_target_idx)
    real_other_expr = load_atlas_expr_rows(sampled_other_idx)
    pred_target_expr = np.asarray(pred_adata.X[sampled_pred_idx], dtype=np.float32)

    real_deg = wilcoxon_deg(real_target_expr, real_other_expr, gene_names)
    pred_deg = wilcoxon_deg(pred_target_expr, real_other_expr, gene_names)
    ct_stats = cell_type_metrics(atlas_obs.iloc[real_target_idx], pred_adata.obs.iloc[pred_target_idx])
    dist_stats = distribution_metrics(real_target_expr, pred_target_expr, seed=SEED + offset)

    metrics_rows.append(
        {
            'time': float(test_time),
            'day': model_to_day[float(test_time)],
            **ct_stats,
            'overlap_up': signature_overlap(real_deg, pred_deg, 'up'),
            'overlap_down': signature_overlap(real_deg, pred_deg, 'down'),
            **dist_stats,
            'n_real_cells': int(real_target_idx.size),
            'n_pred_cells': int(pred_target_idx.size),
            'deg_sample_cells': int(len(sampled_target_idx)),
        }
    )

metrics_df = pd.DataFrame(metrics_rows).sort_values('time').reset_index(drop=True)
metrics_df.to_csv(METRICS_TABLE_PATH, index=False)
display(metrics_df)
print(f'Saved benchmark metrics to {METRICS_TABLE_PATH}')


## Step 3: Load in-memory Navigo predictions and optional baselines onto one shared embedding
The notebook always includes the in-memory Navigo prediction object from Step 1. If external `moscot`, `random`, or `tigon` prediction files are available, they are added to the same embedding for direct comparison at the selected intermediate developmental stage.


In [ ]:
available_methods = {}
missing_methods = []

if FOCUS_TIME is None:
    focus_time = float(test_times[len(test_times) // 2])
else:
    focus_time = float(FOCUS_TIME)
    if focus_time not in {float(t) for t in test_times}:
        raise ValueError(f'FOCUS_TIME={FOCUS_TIME} is not one of the valid held-out times: {test_times}')

available_methods['Navigo'] = subset_for_time(pred_adata, focus_time, MAX_UMAP_CELLS_PER_GROUP, SEED, 'Navigo prediction')

for idx, (method, path) in enumerate(OPTIONAL_BENCHMARK_PREDICTIONS.items()):
    path = Path(path)
    if not path.exists():
        missing_methods.append(method)
        continue
    method_raw = read_h5ad(path)
    method_raw, _, _ = prepare_time_axis(method_raw, prefer_existing_time=True)
    available_methods[method] = subset_for_time(
        method_raw,
        focus_time,
        MAX_UMAP_CELLS_PER_GROUP,
        SEED + 10 + idx,
        method,
    )
    del method_raw

prev_train, next_train = neighboring_train_times(focus_time, train_times)
start_subset = load_atlas_subset_for_time(prev_train, MAX_UMAP_CELLS_PER_GROUP, SEED, 'atlas start stage')
end_subset = load_atlas_subset_for_time(next_train, MAX_UMAP_CELLS_PER_GROUP, SEED + 1, 'atlas end stage')
gt_subset = load_atlas_subset_for_time(focus_time, MAX_UMAP_CELLS_PER_GROUP, SEED + 2, 'atlas intermediate stage')

panel_expr = []
panel_obs = []

def add_panel_block(adata_block: anndata.AnnData, role: str, method: str):
    expr, _ = extract_gene_expression(adata_block, reference_genes=gene_names)
    obs = adata_block.obs.copy().reset_index(drop=True)
    obs['role'] = role
    obs['method'] = method
    obs['day_label'] = obs['day'].astype(str)
    panel_expr.append(expr.astype(np.float32))
    panel_obs.append(obs)

add_panel_block(start_subset, role='observed_start', method='observed')
add_panel_block(end_subset, role='observed_end', method='observed')
add_panel_block(gt_subset, role='ground_truth', method='ground_truth')

for method, adata_method in available_methods.items():
    add_panel_block(adata_method, role='prediction', method=method)

umap_obs = pd.concat(panel_obs, ignore_index=True)
umap_adata = anndata.AnnData(X=np.vstack(panel_expr), obs=umap_obs, var=pd.DataFrame(index=gene_names))
sc.pp.pca(umap_adata, n_comps=min(50, umap_adata.n_obs - 1, umap_adata.n_vars))
sc.pp.neighbors(
    umap_adata,
    n_neighbors=min(30, max(5, umap_adata.n_obs - 1)),
    n_pcs=min(30, umap_adata.obsm['X_pca'].shape[1]),
    random_state=SEED,
)
sc.tl.umap(umap_adata, random_state=SEED, min_dist=0.5)

method_summary = pd.DataFrame(
    {
        'available_methods': list(available_methods.keys()),
        'focus_time': [focus_time] * len(available_methods),
        'focus_day': [model_to_day[focus_time]] * len(available_methods),
        'prev_train_day': [model_to_day[prev_train]] * len(available_methods),
        'next_train_day': [model_to_day[next_train]] * len(available_methods),
    }
)
display(method_summary)
if missing_methods:
    print('Missing optional methods:', ', '.join(missing_methods))


## Step 4: Render the shared UMAP benchmark panels

In [ ]:

prediction_methods = list(available_methods.keys())
n_methods = len(prediction_methods)
ncols = 2 if n_methods > 1 else 1
nrows = math.ceil(n_methods / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 7 * nrows), squeeze=False)
coords = umap_adata.obsm['X_umap']

colors = {
    'observed_start': '#D8E5EA',
    'observed_end': '#C7B8E1',
    'prediction': '#BB7571',
    'ground_truth': '#5A66A0',
}

for panel_idx, method in enumerate(prediction_methods):
    ax = axes[panel_idx // ncols, panel_idx % ncols]

    start_mask = umap_adata.obs['role'] == 'observed_start'
    end_mask = umap_adata.obs['role'] == 'observed_end'
    gt_mask = umap_adata.obs['role'] == 'ground_truth'
    method_mask = (umap_adata.obs['role'] == 'prediction') & (umap_adata.obs['method'] == method)

    ax.scatter(coords[start_mask, 0], coords[start_mask, 1], c=colors['observed_start'], s=5, alpha=0.6, rasterized=True, zorder=1)
    ax.scatter(coords[end_mask, 0], coords[end_mask, 1], c=colors['observed_end'], s=5, alpha=0.6, rasterized=True, zorder=2)
    ax.scatter(coords[method_mask, 0], coords[method_mask, 1], c=colors['prediction'], s=5, alpha=1.0, rasterized=True, zorder=3)
    ax.scatter(coords[gt_mask, 0], coords[gt_mask, 1], c=colors['ground_truth'], s=5, alpha=1.0, rasterized=True, zorder=4)

    ax.set_title(f'{method} (Interpolation Time: {model_to_day[focus_time]})', fontsize=12)
    ax.set_xlabel('UMAP 1', fontsize=10)
    ax.set_ylabel('UMAP 2', fontsize=10)

for extra_idx in range(n_methods, nrows * ncols):
    axes[extra_idx // ncols, extra_idx % ncols].axis('off')

handles = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['observed_start'], markersize=7, label=model_to_day[prev_train]),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['observed_end'], markersize=7, label=model_to_day[next_train]),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['prediction'], markersize=7, label='Prediction'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['ground_truth'], markersize=7, label='Ground truth'),
]
fig.legend(handles=handles, loc='upper center', ncol=4, frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(UMAP_FIG_PATH, dpi=200, bbox_inches='tight')
plt.show()

print(f'Saved UMAP comparison figure to {UMAP_FIG_PATH}')



## Biological interpretation guide
- **Stage reconstruction**: Compare `emd_prediction` against `emd_baseline_source` and `emd_future_stage` to check whether Navigo moves the interpolated stage closer to the true intermediate developmental state than either neighboring stage alone.
- **Cell-type recovery**: Lower Jensen-Shannon and L1 distances indicate that the interpolated population better matches the real intermediate-stage composition.
- **Program recovery**: Higher `overlap_up` and `overlap_down` values indicate stronger agreement between predicted and real intermediate-stage DEG signatures.
- **Geometry recovery**: Lower Wasserstein, MMD, and energy distance values indicate that the predicted cells occupy the same global expression manifold as the real intermediate-stage ground truth.


## Outputs
The notebook keeps the reconstructed Navigo predictions in memory and writes summary tables and figures under `docs/tutorials/resources/interpolation/outputs/04_subtimediff1_benchmark/`.
